# ACE Capital — Facility Underperformance Prediction
**MBA Capstone · Machine Learning Analysis**

**Objective:** Predict which facilities are at risk of EBITDA underperformance *next month* so that operational resources can be deployed proactively.

**Target variable:** `underperformance_flag` (1 = EBITDA < 90% of monthly budget)

**Models compared:** Logistic Regression · Random Forest

**Output:** Risk scores written to `facility_risk_scores` table in Supabase

## 1. Setup — Imports & Supabase Connection

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from dotenv import dotenv_values

from supabase import create_client

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
GOLD = '#b8932a'

# ── Load credentials from ../.env ───────────────────────────────────────────
env = dotenv_values('../.env')
SUPABASE_URL = env.get('VITE_SUPABASE_URL')
SUPABASE_KEY = env.get('VITE_SUPABASE_ANON_KEY')

assert SUPABASE_URL and SUPABASE_KEY, 'Missing Supabase credentials in .env'

sb = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Connected to Supabase ✓')

## 2. Data Pull — Load ML Training Dataset from Supabase

In [ ]:
# Pull the full ML training view (54 columns, 2400 rows)
# vw_ml_training_dataset already contains lagged and rolling features

response = sb.table('vw_ml_training_dataset').select('*').execute()
df = pd.DataFrame(response.data)

# Parse date and sort chronologically within each facility
df['metric_date'] = pd.to_datetime(df['metric_date'])
df = df.sort_values(['facility_id', 'metric_date']).reset_index(drop=True)

print(f'Rows loaded : {len(df):,}')
print(f'Facilities  : {df["facility_id"].nunique()}')
print(f'Date range  : {df["metric_date"].min().date()} → {df["metric_date"].max().date()}')
print(f'Underperf % : {df["underperformance_flag"].mean():.1%}')
df.head(3)

## 3. Feature Engineering — Forward-Shift Target by 1 Month

In [ ]:
# Goal: predict NEXT month's underperformance using THIS month's features.
# Shift the target forward by 1 row within each facility group.

df['target'] = (
    df.groupby('facility_id')['underperformance_flag']
      .shift(-1)   # next month's flag
)

# Drop the last month per facility (no future label available)
df = df.dropna(subset=['target']).copy()
df['target'] = df['target'].astype(int)

# Feature columns — exclude identifiers, the current flag, and date
EXCLUDE = [
    'metric_id', 'facility_id', 'metric_year', 'metric_month',
    'metric_date', 'underperformance_flag', 'target',
    'state', 'facility_type', 'accreditation',  # categorical — encode below
]
CATEGORICALS = ['state', 'facility_type', 'accreditation']

# One-hot encode categoricals
df_enc = pd.get_dummies(df[CATEGORICALS], drop_first=True)
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE] + list(df_enc.columns)

X = pd.concat([df.drop(columns=EXCLUDE + CATEGORICALS), df_enc], axis=1)
X = X.fillna(0)   # lag/roll features are NaN for first rows
y = df['target']

print(f'Features : {X.shape[1]}')
print(f'Samples  : {X.shape[0]:,}')
print(f'Target % : {y.mean():.1%}')

## 4. Train/Test Split — Time-Based
Training on months 1–36 (first 3 years), testing on months 37–48 (final year).
Time-based splitting prevents data leakage — we never train on the future.

In [ ]:
# Cutoff: use metric_date from original df (aligned after dropna)
cutoff = df['metric_date'].sort_values().unique()[35]   # 36th unique month

train_mask = df['metric_date'] <= cutoff
test_mask  = df['metric_date'] >  cutoff

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train: {len(X_train):,} rows  |  cutoff ≤ {cutoff.date()}')
print(f'Test : {len(X_test):,} rows  |  cutoff >  {cutoff.date()}')
print(f'Train underperf %: {y_train.mean():.1%}')
print(f'Test  underperf %: {y_test.mean():.1%}')

## 5. Model 1 — Logistic Regression

In [ ]:
# Logistic Regression with standard scaling (required for regularization)
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_pipe.fit(X_train, y_train)
lr_pred  = lr_pipe.predict(X_test)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred, target_names=['Normal', 'Underperform']))
print(f'AUC: {lr_auc:.4f}')

## 6. Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred, target_names=['Normal', 'Underperform']))
print(f'AUC: {rf_auc:.4f}')

# Feature importance
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
rf_importance.head(15).plot(kind='bar', ax=ax, color=GOLD)
ax.set_title('Random Forest — Top 15 Feature Importances', color='white')
ax.set_ylabel('Importance', color='white')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def metrics_row(name, y_true, y_pred, y_proba):
    return {
        'Model'    : name,
        'Accuracy' : f'{accuracy_score(y_true, y_pred):.3f}',
        'Precision': f'{precision_score(y_true, y_pred):.3f}',
        'Recall'   : f'{recall_score(y_true, y_pred):.3f}',
        'F1'       : f'{f1_score(y_true, y_pred):.3f}',
        'AUC'      : f'{roc_auc_score(y_true, y_proba):.3f}',
    }

comparison = pd.DataFrame([
    metrics_row('Logistic Regression', y_test, lr_pred, lr_proba),
    metrics_row('Random Forest',       y_test, rf_pred, rf_proba),
])
comparison.set_index('Model', inplace=True)
print('=== Model Comparison ===')
display(comparison)

# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba in [('Logistic Regression', lr_proba), ('Random Forest', rf_proba)]:
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
ax.set_title('ROC Curves — Model Comparison', color='white')
plt.tight_layout()
plt.show()

## 8. Risk Score Generation — Score Latest Month Per Facility
Using Random Forest (best performing model) to score each facility's most recent month.

In [ ]:
# Get the latest month row for each facility
latest = (
    df.sort_values('metric_date')
      .groupby('facility_id')
      .last()
      .reset_index()
)

# Re-encode categoricals to match training feature set
latest_enc  = pd.get_dummies(latest[CATEGORICALS], drop_first=True)
X_latest    = pd.concat([latest.drop(columns=EXCLUDE + CATEGORICALS, errors='ignore'), latest_enc], axis=1)
X_latest    = X_latest.reindex(columns=X.columns, fill_value=0).fillna(0)

# Generate probabilities (risk scores) using Random Forest
risk_proba = rf.predict_proba(X_latest)[:, 1]

scores_df = pd.DataFrame({
    'facility_id' : latest['facility_id'].values,
    'metric_year' : latest['metric_year'].values,
    'metric_month': latest['metric_month'].values,
    'risk_score'  : risk_proba,
})

print(f'Scored {len(scores_df)} facilities')
print(f'Avg risk score: {scores_df["risk_score"].mean():.3f}')
scores_df.sort_values('risk_score', ascending=False).head(10)

## 9. Tier Assignment & Distribution

In [ ]:
# Assign risk tiers
def assign_tier(score):
    if score >= 0.65: return 'High'
    if score >= 0.35: return 'Medium'
    return 'Low'

scores_df['risk_tier'] = scores_df['risk_score'].apply(assign_tier)

print('Risk tier distribution:')
print(scores_df['risk_tier'].value_counts())

fig, ax = plt.subplots(figsize=(6, 3))
colors = {'High': '#f87171', 'Medium': '#f59e0b', 'Low': '#34d399'}
tier_counts = scores_df['risk_tier'].value_counts()
tier_counts.plot(kind='bar', ax=ax, color=[colors[t] for t in tier_counts.index])
ax.set_title('Facility Risk Tier Distribution', color='white')
ax.set_ylabel('Count', color='white')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 10. Per-Facility Top Drivers
For each facility, identify the top 3 features contributing most to its risk score.
Computed as: feature importance × |feature value| — highlights which metrics are
both globally important and elevated for that specific facility.

In [ ]:
# Per-facility driver: importance × |normalized feature value|
feature_names = X.columns.tolist()
importances   = rf.feature_importances_  # global RF importances

# Normalize each feature to 0-1 range so magnitudes are comparable
X_latest_arr = X_latest.values
col_max = np.abs(X_latest_arr).max(axis=0)
col_max[col_max == 0] = 1  # avoid divide-by-zero
X_norm = np.abs(X_latest_arr) / col_max

# Driver score = importance × normalized feature value
driver_scores = importances * X_norm  # shape (n_facilities, n_features)

top_drivers = []
for i in range(len(X_latest)):
    sorted_idx = np.argsort(driver_scores[i])[::-1]
    drivers = [feature_names[j] for j in sorted_idx[:3]]
    top_drivers.append(drivers)

scores_df['top_driver_1'] = [d[0] for d in top_drivers]
scores_df['top_driver_2'] = [d[1] for d in top_drivers]
scores_df['top_driver_3'] = [d[2] for d in top_drivers]

print('Sample top drivers for high-risk facilities:')
display(scores_df[scores_df['risk_tier']=='High'][['facility_id','risk_score','top_driver_1','top_driver_2','top_driver_3']].head(5))

## 11. Write Risk Scores to Supabase

In [ ]:
# Prepare payload — round risk_score to 4 decimal places
scores_df['risk_score']    = scores_df['risk_score'].round(4)
scores_df['model_version'] = 'v1'

records = scores_df[[
    'facility_id', 'metric_year', 'metric_month',
    'risk_score', 'risk_tier',
    'top_driver_1', 'top_driver_2', 'top_driver_3',
    'model_version'
]].to_dict(orient='records')

# Upsert — updates existing rows, inserts new ones
result = sb.table('facility_risk_scores').upsert(
    records,
    on_conflict='facility_id,metric_year,metric_month'
).execute()

print(f'✓ Upserted {len(records)} risk score records to Supabase')
print(f'  High risk  : {(scores_df["risk_tier"]=="High").sum()}')
print(f'  Medium risk: {(scores_df["risk_tier"]=="Medium").sum()}')
print(f'  Low risk   : {(scores_df["risk_tier"]=="Low").sum()}')